### Построитель деревьев

**Реализовать класс `TreeBuilder`**. Этот класс призван давать возможность собирать древовидные структуры пошагово.

- Метод `add()` добавляет "лист" в текущий узел дерева:

```python
tree = TreeBuilder()
tree.add('1st')
```
- Свойство `structure` возвращает текущую структуру дерева:

tree.structure  # ['1st']

- Использование экземпляра в качестве менеджера контекста добавляет вложенный узел дерева, делая его текущим в рамках контекста. При этом "спускаться вниз" можно на произвольную глубину:

```python
with tree:
    tree.add('2nd')
    with tree:
        tree.add('3rd')
    tree.add('4th')

tree.structure  # ['1st', ['2nd', ['3rd'], '4th']]
```
- Если в рамках контекста не было добавлено ни одного "листа", то весь узел не должен появляться в итоговой структуре:

```python
tree.structure
['1st', ['2nd', ['3rd'], '4th']]
with tree:
    pass

tree.structure  # ['1st', ['2nd', ['3rd'], '4th']]
# пустой список не был добавлен!
```
**Структура дерева выводится в виде вложенных списков.**

Пример целиком:

```python
tree = TreeBuilder()
tree.structure  # []
tree.add('1st')
tree.structure  # ['1st']
with tree:
    tree.add('2nd')
    with tree:
        tree.add('3rd')
    tree.add('4th')
    with tree:
        pass

tree.structure  # ['1st', ['2nd', ['3rd'], '4th']]
```

---

#### Решение

##### Основная идея

Нужно хранить дерево как вложенные списки, а текущий «активный» узел (куда добавлять элементы) — отдельно. 

**При входе в `with`:**

- Создаём новый пустой список (новый уровень вложенности).
- Добавляем его в текущий активный узел.
- Делаем этот новый список текущим активным узлом.

**При выходе из `with`:**

- Если в этот узел ничего не добавили — удаляем его из родителя.
- Иначе оставляем как есть.

In [1]:
class TreeBuilder:
    """Сборщик деревьев, работающий в виде менеджера контекста."""

    def __init__(self):
        # Корневой список, в который будут добавляться элементы
        self._root = []
        # Текущий активный узел (список), куда добавляем элементы
        self._current = self._root

    def __enter__(self):
        # Создаём новый вложенный список
        new_node = []
        # Добавляем его в текущий узел
        self._current.append(new_node)
        # Делаем его текущим
        self._current = new_node
        return self

    def __exit__(self, exc_type, exc, tb):
        # Если в текущем узле ничего не добавили, удаляем его из родителя
        if not self._current:
            # Находим родителя: это тот список, который был текущим ДО входа в контекст.
            # Но у нас нет прямой ссылки на родителя, поэтому нужно найти, где лежит self._current
            # в структуре дерева. Проще всего хранить стек родителей.
            pass
        # Возвращаем False, чтобы исключения не подавлялись
        return False

##### Проблема

В такой реализации мы не можем легко удалить пустой узел при выходе, потому что не помним, какой список был «родителем». Поэтому лучше хранить стек активных узлов.

---

In [1]:
class TreeBuilder:
    """Сборщик деревьев, работающий в виде менеджера контекста."""

    def __init__(self):
        self._root = []          # Корневая структура
        self._stack = [self._root]  # Стек: каждый элемент — текущий активный список

    def __enter__(self):
        # Создаём новый узел (пустой список)
        new_node = []
        # Добавляем в текущий (верхний в стеке)
        self._stack[-1].append(new_node)
        # Теперь этот узел становится текущим
        self._stack.append(new_node)
        return self

    def __exit__(self, exc_type, exc, tb):
        # Удаляем текущий узел из стека
        current = self._stack.pop()
        # Если он пустой — удаляем из родителя
        if not current:
            parent = self._stack[-1]
            # current — это последний элемент parent, потому что мы только что его туда добавили
            parent.pop()
        return False

    def add(self, value):
        """Добавляет значение в текущую позицию в дереве."""
        # Просто добавляем в текущий активный список (верх стека)
        self._stack[-1].append(value)

    @property
    def structure(self):
        """Возвращает текущую структуру дерева в виде вложенных списков."""
        return self._root

##### Разбор ключевых моментов

- `_stack`: хранит цепочку вложенных списков от корня до текущего. `_stack[-1]` — это всегда тот список, куда нужно добавлять.

- `__enter__`: создаёт новый список, добавляет его в текущий, и делает его новым текущим.

- `__exit__`: убирает текущий из стека; если он пуст — удаляет из родителя, иначе оставляет.

- `add`: просто делает append в текущий список (в `_stack[-1]`).

- `structure`: возвращает `_root`, потому что вся структура уже там.
---